# Module 8 | Class 3 -- Flask Prediction API

**Objective:** Package a trained ML model into a REST API using Flask, so other applications can send data and receive predictions over HTTP.

**Important:** Flask does not run as a server inside Google Colab. This notebook **prepares all the files** you need, then you run them locally on your machine.

What we will create:
1. A trained model saved with `joblib`
2. `app.py` -- the Flask API server
3. `requirements.txt` -- dependencies
4. Test code to verify the API works

## 1. Setup and Train a Model

In [ ]:
!pip install -q scikit-learn pandas joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42

# Create output directory
os.makedirs('flask_api', exist_ok=True)

print("Setup complete.")

In [ ]:
# Load and preprocess Telco Churn data (same as Module 8 Class 2)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df_proc = df.copy()
df_proc['TotalCharges'] = pd.to_numeric(df_proc['TotalCharges'], errors='coerce')
df_proc['TotalCharges'].fillna(df_proc['TotalCharges'].median(), inplace=True)
df_proc.drop('customerID', axis=1, inplace=True)
df_proc['Churn'] = (df_proc['Churn'] == 'Yes').astype(int)

cat_cols = df_proc.select_dtypes(include='object').columns.tolist()
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col])
    le_dict[col] = le

X = df_proc.drop('Churn', axis=1)
y = df_proc['Churn']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data ready: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features ({len(feature_names)}): {feature_names}")

In [ ]:
# Train the model
model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

## 2. Save Model and Scaler

In [ ]:
# Save the trained model and scaler
joblib.dump(model, 'flask_api/model.joblib')
joblib.dump(scaler, 'flask_api/scaler.joblib')

# Save feature names for validation
with open('flask_api/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print("Saved:")
print("  flask_api/model.joblib")
print("  flask_api/scaler.joblib")
print("  flask_api/feature_names.json")

## 3. Create app.py (Flask API)

The `%%writefile` magic writes the cell content directly to a file.

In [ ]:
%%writefile flask_api/app.py
"""Flask API for Telco Churn prediction."""

from flask import Flask, request, jsonify
import joblib
import json
import numpy as np

app = Flask(__name__)

# Load model, scaler, and feature names at startup
model = joblib.load('model.joblib')
scaler = joblib.load('scaler.joblib')
with open('feature_names.json') as f:
    FEATURE_NAMES = json.load(f)


@app.route('/')
def home():
    return jsonify({
        'service': 'Telco Churn Prediction API',
        'endpoints': {
            '/predict': 'POST - predict churn for a single customer',
            '/info': 'GET - model info and expected features'
        }
    })


@app.route('/info')
def info():
    return jsonify({
        'model_type': type(model).__name__,
        'n_features': len(FEATURE_NAMES),
        'features': FEATURE_NAMES
    })


@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()

        if 'features' not in data:
            return jsonify({'error': 'Missing "features" key in request body'}), 400

        features = data['features']

        if len(features) != len(FEATURE_NAMES):
            return jsonify({
                'error': f'Expected {len(FEATURE_NAMES)} features, got {len(features)}'
            }), 400

        # Scale and predict
        X = np.array(features).reshape(1, -1)
        X_scaled = scaler.transform(X)
        prediction = int(model.predict(X_scaled)[0])
        probability = float(model.predict_proba(X_scaled)[0][1])

        return jsonify({
            'prediction': prediction,
            'churn': bool(prediction),
            'churn_probability': round(probability, 4)
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    print(f"Model loaded: {type(model).__name__}")
    print(f"Features: {len(FEATURE_NAMES)}")
    app.run(host='0.0.0.0', port=5000, debug=True)

## 4. Create requirements.txt

In [ ]:
%%writefile flask_api/requirements.txt
flask>=2.0
scikit-learn>=1.0
joblib>=1.0
numpy>=1.21
requests>=2.25

## 5. Verify Files Were Created

In [ ]:
import os

print("Files in flask_api/:")
for f in sorted(os.listdir('flask_api')):
    size = os.path.getsize(f'flask_api/{f}')
    print(f"  {f:<25s} {size:>8,} bytes")

## 6. How to Run Locally

Download the `flask_api/` folder from Colab, then run:

```bash
cd flask_api
pip install -r requirements.txt
python app.py
```

The server will start on `http://localhost:5000`.

## 7. Test Code

Once the server is running, use this code to test it. (You can also run this from a second terminal.)

In [ ]:
# This cell is for testing AFTER you run the Flask server locally.
# It will fail in Colab since the server is not running here.

import requests
import json

BASE_URL = "http://localhost:5000"

# Test 1: Home endpoint
print("--- GET / ---")
# response = requests.get(f"{BASE_URL}/")
# print(response.json())

# Test 2: Info endpoint
print("\n--- GET /info ---")
# response = requests.get(f"{BASE_URL}/info")
# print(json.dumps(response.json(), indent=2))

# Test 3: Predict endpoint
print("\n--- POST /predict ---")
# Use a sample from the test set
sample_features = X_test.iloc[0].tolist()
actual_label = y_test.iloc[0]
print(f"Sample features (first 5): {sample_features[:5]}")
print(f"Actual label: {actual_label}")

# Uncomment to test when server is running:
# response = requests.post(
#     f"{BASE_URL}/predict",
#     json={"features": sample_features}
# )
# print(response.json())

print("\n(Uncomment the requests lines above after starting the Flask server locally.)")

In [ ]:
# Generate a curl command for testing from terminal
sample = X_test.iloc[0].tolist()
payload = json.dumps({"features": sample})

print("Test with curl (run in terminal after starting the server):")
print()
print(f"curl -X POST http://localhost:5000/predict \\")
print(f"  -H 'Content-Type: application/json' \\")
print(f"  -d '{payload}'")

---
## TODO: Student Exercise

**Task 1:** Add a `/health` endpoint to `app.py` that returns:
```json
{"status": "healthy", "model_loaded": true}
```

**Task 2:** Modify the `/predict` endpoint to also return class probabilities for both classes:
```json
{
    "prediction": 0,
    "churn": false,
    "churn_probability": 0.15,
    "probabilities": {"no_churn": 0.85, "churn": 0.15}
}
```

**Task 3:** Write the updated `app.py` using `%%writefile` and test it.

In [ ]:
# TODO: Write your updated app.py here
# %%writefile flask_api/app_v2.py

# Your code here


In [ ]:
# TODO: Write test code for your new endpoints
